# Re-run: the loop over a course (Paper VI, RQ4)

RQ4 is a simulation, so this notebook has two parts with different inputs.

1. **From the per-case file alone** (Python 3.9+ and numpy; `per-case-model-outputs.jsonl` or the `.gz`
   the report offers in this folder, or in a folder named by `RELEASE_DIR`): the label effect each
   model's stand-in grader is fitted with (0.093, 0.100, 0.140, 0.199 marks per point of label) and the
   arithmetic of the loop that follows from it.
2. **With the release's code package as well.** The table's simulated columns come from
   `studies/measurement/grader-label-compounding/compounding.py` and its committed `out/params.json` and
   `out/floor.json`, published with the release under `code/`. Put that `code/` folder beside this
   notebook (or set `RELEASE_DIR` to the release directory that holds it). Part 2 imports
   `compounding.py` and `code/shared/seeds.py` and nothing else; it stops with a message naming the
   missing file if the package is not there. It re-runs the primary configuration for the four models,
   about half a minute in all.

The fit (part 1) needs no corpus. `params.json` also records the cohort's shape, which `compounding.py`
read from the JorGPT corpus when it wrote the file; the notebook uses the committed values and does
not read the corpus.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grader-reliability-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
import importlib.util
import json
import math
import numpy as np

RUNS = {"Gemini 3.5 Flash Lite": "grader-vi/swap-gemini", "GPT-4.1-mini": "grader-vi/swap-gpt41mini",
        "GPT-5-mini": "grader-vi/swap-gpt5mini", "Qwen3-235B": "grader-vi/swap-qwen"}
MODEL_ID = {"Gemini 3.5 Flash Lite": "google:gemini-3.5-flash-lite", "GPT-4.1-mini": "openai:gpt-4.1-mini",
            "GPT-5-mini": "openai:gpt-5-mini", "Qwen3-235B": "openrouter:qwen/qwen3-235b-a22b-2507"}

def same(x, printed, nd):
    return f"{x:.{nd}f}" == f"{printed:.{nd}f}"

## Part 1: the fitted label effect, from the per-case file

The stand-in grader's label effect **gamma** is the slope of (arm mark − control mark) on (label − 5),
with (teacher mark − 5) and a constant beside it, by least squares over the true-label and the
swapped-label arms stacked (600 rows per model). The swap makes the label vary on identical work, so
the slope is the label's causal effect. The calibration (a, b) is the control mark regressed on the
teacher's mark. This is `fit_grader` in `compounding.py`, written out.

In [3]:
def fit(rows):
    ok = [r for r in rows if r["control"] is not None]
    h = np.array([r["human"] for r in ok], float)
    c = np.array([r["control"] for r in ok], float)
    (a, b), *_ = np.linalg.lstsq(np.column_stack([np.ones_like(h), h]), c, rcond=None)
    y, l5, h5 = [], [], []
    for r in ok:
        for arm, lab in (("trueLabel", "labelTrue"), ("swappedLabel", "labelSwapped")):
            if r[arm] is not None:
                y.append(r[arm] - r["control"]); l5.append(r[lab] - 5.0); h5.append(r["human"] - 5.0)
    X = np.column_stack([np.ones(len(y)), l5, h5])
    (delta, gamma, eta), *_ = np.linalg.lstsq(X, np.array(y, float), rcond=None)
    return {"a": a, "b": b, "delta": delta, "gamma": gamma, "eta": eta, "rows": len(y)}

fits = {m: fit(list(per_case({run}))) for m, run in RUNS.items()}
for m, f in fits.items():
    print(f"{m:22s} gamma {f['gamma']:.3f} marks per label point   (a {f['a']:.2f}, b {f['b']:.2f}, "
          f"delta {f['delta']:+.3f}, eta {f['eta']:+.3f}; {f['rows']} rows)")
for m, want in zip(RUNS, (0.093, 0.100, 0.140, 0.199)):
    assert same(fits[m]["gamma"], want, 3), (m, fits[m]["gamma"], want)

Gemini 3.5 Flash Lite  gamma 0.093 marks per label point   (a 1.28, b 0.82, delta -0.022, eta -0.012; 600 rows)
GPT-4.1-mini           gamma 0.100 marks per label point   (a 1.41, b 0.56, delta -0.154, eta -0.022; 598 rows)
GPT-5-mini             gamma 0.140 marks per label point   (a 1.83, b 0.57, delta -0.285, eta -0.029; 597 rows)
Qwen3-235B             gamma 0.199 marks per label point   (a 1.56, b 0.52, delta -0.071, eta -0.074; 585 rows)


**The loop's arithmetic.** On the shipped engine a skill is revisited every 10 days (four skills, one
item every 2.5 days), so its belief decays toward the prior by r = (1 + 0.1 × 10)^−0.5 between writes.
In steady state one grade moves the belief by w = M / n\* of its score, with n\* = 1 + M / (1 − r) and
evidence mass M = 1. Without the edge the deviation of a belief decays with eigenvalue r; with the edge
it is r + w·gamma, and the loop runs away when gamma reaches gamma\* = 1 + (1 − r) / M.

In [4]:
M = 1.0
r = (1 + 0.1 * 2.5 * 4) ** -0.5
n_star = 1 + M / (1 - r)
w = M / n_star
gamma_star = 1 + (1 - r) / M
print(f"r = eigenvalue without the edge {r:.3f}; one grade moves the belief by w = {w:.2f}; runaway at gamma* = {gamma_star:.2f}")
assert same(r, 0.707, 3) and same(w, 0.23, 2) and same(gamma_star, 1.29, 2)
fitted_multiple = {m: gamma_star / f["gamma"] for m, f in fits.items()}
print("runaway multiple against the fitted effect:", {m: f"{v:.1f}×" for m, v in fitted_multiple.items()})
for m, want in zip(RUNS, (13.8, 13.0, 9.3, 6.5)):
    assert same(fitted_multiple[m], want, 1), (m, fitted_multiple[m], want)
print("Part 1: the four fitted effects, r, w, gamma* and the four fitted-effect multiples match the paper.")

r = eigenvalue without the edge 0.707; one grade moves the belief by w = 0.23; runaway at gamma* = 1.29
runaway multiple against the fitted effect: {'Gemini 3.5 Flash Lite': '13.8×', 'GPT-4.1-mini': '13.0×', 'GPT-5-mini': '9.3×', 'Qwen3-235B': '6.5×'}
Part 1: the four fitted effects, r, w, gamma* and the four fitted-effect multiples match the paper.


## Part 2: the simulation, from the release's code package

The table's eigenvalues and runaway multiples use the **latent** label effect: the fitted slope is
attenuated because marks are clipped to 0–10, and the study's detection floor measured that attenuation
(`out/floor.json`, `attenuation_curve`), which is inverted here. The loop multiplier, belief error,
strong learners' under-belief and recovery come from re-running the simulation with the committed
parameters and the study's own seeds; the simulation is deterministic, so the re-run reproduces the
committed results exactly.

In [5]:
def code_file(rel):
    """A file of the release's code package: code/<rel> beside this notebook, in RELEASE_DIR or a checkout."""
    try:
        return find_release_file("code/" + rel)
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Part 2 needs the release's code package, and code/{rel} was not found. Download the code "
            f"package of release {RELEASE_ID} (its location and hash are in the release file) and put its "
            f"code/ folder beside this notebook, or set RELEASE_DIR to the release directory that holds it. "
            f"Part 1 above needs only the data file.") from None

study = code_file("studies/measurement/grader-label-compounding/compounding.py").parent
code_file("shared/seeds.py")
spec = importlib.util.spec_from_file_location("compounding", study / "compounding.py")
C = importlib.util.module_from_spec(spec)
spec.loader.exec_module(C)          # compounding.py puts code/ on sys.path for shared.seeds
prm = json.loads((study / "out/params.json").read_text())
curve = json.loads((study / "out/floor.json").read_text())["attenuation_curve"]
committed = json.loads((study / "out/results.json").read_text())["per_model"]
cohort, lab = prm["cohort"], prm["tail_labels"]
print(f"cohort: {cohort['T']} items per term, ability mean {cohort['mu_theta']:.2f}, sd {cohort['sd_theta']:.2f}")

# The committed parameters are the fit of part 1.
for m in RUNS:
    P = prm["models"][MODEL_ID[m]]["fit"]
    assert all(abs(P[k] - fits[m][k]) < 1e-9 for k in ("a", "b", "delta", "gamma", "eta")), m
print("params.json holds exactly the fit recomputed in part 1.")

cohort: 40 items per term, ability mean 5.05, sd 1.27
params.json holds exactly the fit recomputed in part 1.


In [6]:
latent = {m: C.delatent(fits[m]["gamma"], curve) for m in RUNS}
lam = {m: r + w * latent[m] for m in RUNS}
multiple = {m: gamma_star / latent[m] for m in RUNS}
for m in RUNS:
    print(f"{m:22s} latent effect {latent[m]:.3f}   largest eigenvalue {lam[m]:.3f}   runaway at {multiple[m]:.1f}× its own")
for m, (lt, ev, mu) in zip(RUNS, [(0.103, 0.730, 12.6), (0.110, 0.732, 11.8), (0.154, 0.742, 8.4), (0.220, 0.757, 5.9)]):
    assert same(latent[m], lt, 3) and same(lam[m], ev, 3), (m, latent[m], lam[m])
    # The printed multiple is computed from the unrounded critical effect and latent effect. (Until
    # 2026-10-06 the paper printed 12.5 and 11.7, from the rounded 1.29 and latent effects: corrected.)
    assert same(multiple[m], mu, 1), (m, multiple[m], mu)
    print(f"{m:22s} printed {mu}×  = {gamma_star:.4f} / {latent[m]:.4f} = {multiple[m]:.2f}")

Gemini 3.5 Flash Lite  latent effect 0.103   largest eigenvalue 0.730   runaway at 12.6× its own
GPT-4.1-mini           latent effect 0.110   largest eigenvalue 0.732   runaway at 11.8× its own
GPT-5-mini             latent effect 0.154   largest eigenvalue 0.742   runaway at 8.4× its own
Qwen3-235B             latent effect 0.220   largest eigenvalue 0.757   runaway at 5.9× its own
Gemini 3.5 Flash Lite  printed 12.6×  = 1.2929 / 0.1028 = 12.57
GPT-4.1-mini           printed 11.8×  = 1.2929 / 0.1100 = 11.75
GPT-5-mini             printed 8.4×  = 1.2929 / 0.1544 = 8.37
Qwen3-235B             printed 5.9×  = 1.2929 / 0.2200 = 5.88


In [7]:
sim = {}
for m in RUNS:
    mid = MODEL_ID[m]
    P = prm["models"][mid]["fit"]
    cfg = C.CONFIGS["primary"]
    c2 = dict(cohort)
    mg = np.clip((P["a"] + P["b"] * cohort["mu_theta"]) / 10.0, 0.02, 0.98)
    c2["mu_logit"] = float(math.log(mg / (1 - mg)))
    sim[m] = C.evaluate(P, prm["models"][mid]["draws"], cfg, c2, lab, C.N_POINT, C.N_DRAW,
                        C.arm_seed(C.SEED, f"{mid}-primary"), curve=curve)
    pt, ci, rec = sim[m]["point"], sim[m]["ci95"], sim[m]["recovery"]
    print(f"{m:22s} multiplier {pt['A']:.3f} [{ci['A'][0]:.3f}, {ci['A'][1]:.3f}]  total {pt['A_total']:.3f}  "
          f"belief error {pt['dRMSE']:+.3f}  strong {pt['err_noedge_strong']:+.3f} → {pt['err_edge_strong']:+.3f}  "
          f"recovery {rec['point']['rec_median_noedge']:.0f} → {rec['point']['rec_median_edge']:.0f} items  "
          f"(ratio interval [{rec['ci95']['R'][0]:.2f}, {rec['ci95']['R'][1]:.2f}])")

Gemini 3.5 Flash Lite  multiplier 1.066 [1.049, 1.083]  total 1.053  belief error -0.002  strong -0.046 → -0.040  recovery 23 → 25 items  (ratio interval [1.04, 1.13])
GPT-4.1-mini           multiplier 1.075 [1.061, 1.091]  total 1.032  belief error +0.010  strong -0.160 → -0.175  recovery 19 → 20 items  (ratio interval [1.00, 1.11])
GPT-5-mini             multiplier 1.111 [1.093, 1.128]  total 1.054  belief error +0.010  strong -0.127 → -0.150  recovery 21 → 23 items  (ratio interval [1.05, 1.10])
Qwen3-235B             multiplier 1.167 [1.149, 1.183]  total 1.005  belief error +0.012  strong -0.170 → -0.187  recovery 19 → 22 items  (ratio interval [1.11, 1.21])


In [8]:
# The paper's printed values (RQ4 table and text). Point estimates must match exactly.
PRINTED = {  # multiplier, belief error change, strong under-belief without → with, recovery without → with, total
    "Gemini 3.5 Flash Lite": (1.066, -0.002, -0.046, -0.040, 23, 25, 1.053),
    "GPT-4.1-mini": (1.075, 0.010, -0.160, -0.175, 19, 20, 1.032),
    "GPT-5-mini": (1.111, 0.010, -0.127, -0.150, 21, 23, 1.054),
    "Qwen3-235B": (1.167, 0.012, -0.170, -0.187, 19, 22, 1.005),
}
for m, (A, dr, s0, s1, r0, r1, At) in PRINTED.items():
    pt, rp = sim[m]["point"], sim[m]["recovery"]["point"]
    assert same(pt["A"], A, 3) and same(pt["A_total"], At, 3), (m, pt["A"], pt["A_total"])
    assert same(pt["dRMSE"], dr, 3), (m, pt["dRMSE"])
    assert same(pt["err_noedge_strong"], s0, 3) and same(pt["err_edge_strong"], s1, 3), m
    assert (rp["rec_median_noedge"], rp["rec_median_edge"]) == (r0, r1), (m, rp)
    # and the re-run equals the committed results.json, to floating-point agreement: a different
    # machine's numerical library can differ in the last bits, which no printed digit can show
    assert math.isclose(pt["A"], committed[MODEL_ID[m]]["primary"]["point"]["A"], rel_tol=1e-9, abs_tol=1e-12), \
        (m, pt["A"], committed[MODEL_ID[m]]["primary"]["point"]["A"])
# "Recovery is resolved as slower on three of the four models": the ratio's interval excludes 1.
slower = [m for m in RUNS if sim[m]["recovery"]["ci95"]["R"][0] > 1]
assert slower == ["Gemini 3.5 Flash Lite", "GPT-5-mini", "Qwen3-235B"], slower
PRINTED_CI = {"Gemini 3.5 Flash Lite": (1.049, 1.083), "GPT-4.1-mini": (1.061, 1.091),
              "GPT-5-mini": (1.093, 1.128), "Qwen3-235B": (1.149, 1.183)}
same_ci = all(same(sim[m]["ci95"]["A"][i], PRINTED_CI[m][i], 3) for m in RUNS for i in (0, 1))
print("All 28 simulated point values in the table and text match the paper; recovery is slower on", ", ".join(slower))
print("Multiplier intervals equal the printed ones:", same_ci)

All 28 simulated point values in the table and text match the paper; recovery is slower on Gemini 3.5 Flash Lite, GPT-5-mini, Qwen3-235B
Multiplier intervals equal the printed ones: True
